In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os


In [ ]:
import pandas as pd
import io

def excel_clipboard_to_dataframe():
    """
    Копируете таблицу в Excel (Ctrl+C), 
    затем запускаете эту функцию
    """
    try:
        # Читаем из буфера обмена
        df = pd.read_clipboard()
        print(f"✅ Успешно загружено из буфера обмена: {df.shape}")
        return df
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return None

# Использование:
# 1. Откройте Excel файл
# 2. Выделите и скопируйте таблицу (Ctrl+A, Ctrl+C)  
# 3. Запустить:
df = excel_clipboard_to_dataframe()
if df is not None:
    print(df.head())

✅ Успешно загружено из буфера обмена: (32, 13)
   Число Месяц Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6  \
0    NaN     1          2          3          4          5          6   
1    1.0  8,87       8,64         10       12 _       63,3         60   
2    2.0  8,64     9.11 ^       9,06       18,9       50,9     54.9 _   
3    3.0  8,87       9,11       8,61       16,2         49       63,3   
4    4.0  8,64       9,34       9,28       12,4         50       62,2   

  Unnamed: 7 Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12  
0          7          8          9          10          11          12  
1       93,2       73,1       44,2      30.9 ^      23.1 ^      16.9 ^  
2       90,8         72       44,2        27,8        22,2        15,5  
3       80,7         72       42,3        29,3        22,2        13,4  
4       80,7     74.2 ^       43,3        27,8      23.1 ^        11,8  


In [ ]:
import pandas as pd
import os
from datetime import datetime
import numpy as np

def excel_clipboard_to_dataframe_improved(verbose=True):
    """
    Улучшенная функция для чтения таблиц из буфера обмена
    """
    try:
        # Читаем сырые данные
        raw_data = pd.read_clipboard(sep='\t', header=None)
        
        if verbose:
            print(f"Сырые данные загружены: {raw_data.shape}")
            print("Первые 5 строк сырых данных:")
            print(raw_data.head())
            print("-" * 50)
        
        return raw_data
        
    except Exception as e:
        print(f"Ошибка при чтении из буфера обмена: {e}")
        return None

def clean_and_prepare_data(raw_df, verbose=True):
    """
    Очищает и подготавливает данные к обработке
    """
    try:
        # Создаем копию для работы
        df = raw_df.copy()
        
        if verbose:
            print("🔧 Начинаем очистку данных...")
        
        # Ищем строку с названиями месяцев (обычно это вторая строка)
        month_row_idx = None
        for i in range(min(3, len(df))):  # Проверяем первые 3 строки
            row_values = df.iloc[i].dropna().astype(str).tolist()
            # Ищем строку, где есть числа от 1 до 12
            month_numbers = [x for x in row_values if x.replace('.', '').isdigit() and 1 <= int(float(x)) <= 12]
            if len(month_numbers) >= 6:  # Если найдено достаточно месяцев
                month_row_idx = i
                break
        
        if month_row_idx is None:
            # Если не нашли месяцы, используем первую строку как заголовок
            df.columns = df.iloc[0]
            df = df[1:].reset_index(drop=True)
            month_row_idx = 0
        else:
            # Используем найденную строку как заголовок
            df.columns = df.iloc[month_row_idx]
            df = df[month_row_idx + 1:].reset_index(drop=True)
        
        # Переименовываем первый столбец в 'Число'
        if df.columns[0] != 'Число':
            df = df.rename(columns={df.columns[0]: 'Число'})
        
        # Оставляем только числовые столбцы (месяцы) и столбец 'Число'
        numeric_cols = []
        for col in df.columns:
            if col == 'Число' or (str(col).replace('.', '').isdigit() and 1 <= int(float(col)) <= 12):
                numeric_cols.append(col)
        
        df = df[numeric_cols]
        
        # Очищаем данные от лишних символов
        for col in df.columns:
            if col != 'Число':
                df[col] = df[col].astype(str).str.replace('^', '', regex=False).str.replace('"', '', regex=False)
                df[col] = df[col].str.replace(',', '.').str.strip()
                # Заменяем пустые строки и некорректные значения на NaN
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Преобразуем столбец 'Число' в числовой формат
        df['Число'] = pd.to_numeric(df['Число'], errors='coerce')
        
        # Удаляем строки, где нет чисел (дней)
        df = df.dropna(subset=['Число']).reset_index(drop=True)
        
        if verbose:
            print(f"Данные очищены: {df.shape}")
            print("Структура очищенных данных:")
            print(df.head())
            print(f"Столбцы: {list(df.columns)}")
            print("-" * 50)
        
        return df
        
    except Exception as e:
        print(f"Ошибка при очистке данных: {e}")
        return None

def process_year_data_improved(df, year, value_name='Расход', verbose=True):
    """
    Улучшенная функция преобразования таблицы в длинный формат
    """
    try:
        if verbose:
            print(f"Обработка {year} года...")
            print(f"Доступные стодбцы: {list(df.columns)}")
        
        # Создаем список месяцев из названий столбцов
        months = [col for col in df.columns if col != 'Число' and str(col).replace('.', '').isdigit()]
        
        if verbose:
            print(f"Найдены месяцы: {months}")
        
        # Преобразуем в длинный формат
        long_df = df.melt(
            id_vars=['Число'],
            value_vars=months,
            var_name='Месяц',
            value_name=value_name
        )
        
        if verbose:
            print(f"Данные после melt: {long_df.shape}")
            print(f"Пример данных:")
            print(long_df.head())
        
        # Преобразуем месяцы в числа
        long_df['Месяц'] = pd.to_numeric(long_df['Месяц'], errors='coerce')
        
        def create_date(row):
            try:
                day = int(float(row['Число']))
                month = int(float(row['Месяц']))
                # Проверяем валидность даты
                if 1 <= day <= 31 and 1 <= month <= 12:
                    return datetime(year, month, day)
                return pd.NaT
            except:
                return pd.NaT
        
        long_df['Дата'] = long_df.apply(create_date, axis=1)
        
        # Удаляем строки с некорректными датами или пустыми значениями
        initial_count = len(long_df)
        long_df = long_df.dropna(subset=['Дата', value_name])
        final_count = len(long_df)
        
        if verbose:
            print(f"Удалено пустых строк: {initial_count - final_count}")
            print(f"Осталось записей: {final_count}")
        
        if final_count == 0:
            print("!!После обработки не осталось данных!")
            print("Проверка первых 10 строк исходных данных:")
            print(long_df.head(10))
            return None
        
        # Сортируем по дате и выбираем нужные столбцы
        result_df = long_df[['Дата', value_name]].sort_values('Дата').reset_index(drop=True)
        result_df['Год'] = year
        
        if verbose:
            print(f"Обработано записей: {len(result_df)}")
            print(f"Период: {result_df['Дата'].min().strftime('%Y-%m-%d')} - {result_df['Дата'].max().strftime('%Y-%m-%d')}")
            print(f"Пример результата:")
            print(result_df.head())
        
        return result_df
        
    except Exception as e:
        print(f"Ошибка при обработке данных за {year} год: {e}")
        import traceback
        print(f"Детали ошибки: {traceback.format_exc()}")
        return None

def process_single_year_interactive(year, value_name='Расход'):
    """
    Интерактивная обработка одного года с улучшенной диагностикой
    """
    print(f"\nОБРАБОТКА {year} ГОДА")
    print("=" * 60)
    
    input(f"Аодготовьте таблицу за {year} год в Excel, скопируйте её (Ctrl+A, Ctrl+C) и нажмите Enter...")
    
    # Читаем сырые данные
    raw_data = excel_clipboard_to_dataframe_improved(verbose=True)
    
    if raw_data is None:
        print(f"Не удалось загрузить данные за {year} год")
        return None
    
    # Очищаем данные
    cleaned_data = clean_and_prepare_data(raw_data, verbose=True)
    
    if cleaned_data is None:
        print(f"Не удалось очистить данные за {year} год")
        return None
    
    # Обрабатываем данные
    processed_data = process_year_data_improved(cleaned_data, year, value_name, verbose=True)
    
    return processed_data

def process_multiple_years_improved(years_range, value_name='Расход'):
    """
    Улучшенная обработка данных за несколько лет
    """
    all_data = []
    successful_years = []
    failed_years = []
    
    print("УЛУЧШЕННЫЙ СБОР ДАННЫХ ЗА НЕСКОЛЬКО ЛЕТ")
    print("=" * 60)
    
    for year in years_range:
        processed_data = process_single_year_interactive(year, value_name)
        
        if processed_data is not None and len(processed_data) > 0:
            all_data.append(processed_data)
            successful_years.append(year)
            print(f"{year} год успешно обработан ({len(processed_data)} записей)")
        else:
            failed_years.append(year)
            print(f"{year} год не обработан")
        
        print("-" * 60)
    
    if all_data:
        # Объединяем все данные
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_df = combined_df.sort_values('Дата').reset_index(drop=True)
        
        print(f"\nОБРАБОТКА ЗАВЕРШЕНА!")
        print("=" * 60)
        print(f"ИТОГОВАЯ СТАТИСТИКА:")
        print(f"Успешно обработано лет: {len(successful_years)}")
        print(f"Успешные годы: {successful_years}")
        if failed_years:
            print(f"Пропущенные годы: {failed_years}")
        print(f"Общее количество записей: {len(combined_df):,}")
        print(f"Период данных: {combined_df['Дата'].min().strftime('%Y-%m-%d')} - {combined_df['Дата'].max().strftime('%Y-%m-%d')}")
        print(f"Среднее значение: {combined_df[value_name].mean():.2f}")
        
        return combined_df
    else:
        print("Не удалось обработать ни один год")
        return None

# Пример использования
if __name__ == "__main__":
    # Определяем диапазон лет
    years = list(range(2008, 2018)) + list(range(2019, 2024))
    
    print(f"ОБРАБОТАТЬ ДАННЫЕ ЗА {len(years)} ЛЕТ ({years[0]}-{years[-1]})")
    
    # Обрабатываем все годы
    combined_data = process_multiple_years_improved(years, value_name='Расход')
    
    if combined_data is not None:
        # Сохраняем результаты
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        """ # Сохраняем в CSV
        csv_file = f"расходы_полные_данные_{timestamp}.csv"
        combined_data.to_csv(csv_file, index=False, encoding='utf-8-sig')
        print(f"Данные сохранены в CSV: {csv_file}") """
        
        # Сохраняем в Excel
        hydro_post = input("Введите номер гидрологического поста")
        excel_file = f"расходы_полные_данные_{timestamp}_{hydro_post}.xlsx"
        combined_data.to_excel(excel_file, index=False)
        print(f"Данные сохранены в Excel: {excel_file}")
        
        # Дополнительная статистика
        print(f"\nСТАТИСТИКА ПО ГОДАМ:")
        yearly_stats = combined_data.groupby('Год').agg({
            'Расход': ['count', 'mean', 'min', 'max']
        }).round(2)
        yearly_stats.columns = ['Количество', 'Среднее', 'Минимум', 'Максимум']
        print(yearly_stats)
        
    else:
        print("Программа завершена с ошибками.")

🎯 ЦЕЛЬ: ОБРАБОТАТЬ ДАННЫЕ ЗА 15 ЛЕТ (2008-2023)
🚀 УЛУЧШЕННЫЙ СБОР ДАННЫХ ЗА НЕСКОЛЬКО ЛЕТ

📅 ОБРАБОТКА 2008 ГОДА
✅ Сырые данные загружены: (33, 13)
Первые 5 строк сырых данных:
      0      1       2      3    4      5    6    7      8     9     10  \
0  Число  Месяц     NaN    NaN  NaN    NaN  NaN  NaN    NaN   NaN   NaN   
1    NaN      1       2      3    4    5.0    6  7.0      8     9    10   
2      1  121 ^  81.2 _  135 _  175  191.0  NaN  NaN    198  62,8  70,2   
3      2    120    82,6    138  172  181.0  NaN  NaN  206 ^  62,4  69,8   
4      3    120    83,9    142  168  172.0  NaN  NaN    199    62  69,5   

       11    12  
0     NaN   NaN  
1      11    12  
2  75.4 _  83,2  
3    76,8    84  
4    78,3  84,8  
--------------------------------------------------
🔧 Начинаем очистку данных...
✅ Данные очищены: (31, 13)
Структура очищенных данных:
1  Число      1     2      3      4    5.0   6  7.0      8     9    10    11  \
0      1  121.0   NaN    NaN  175.0  191.0 NaN  N